<a href="https://colab.research.google.com/github/Emmanuel-NS/Rl_Mission-based_Summative/blob/main/colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FARUMASI rider delivery

Use a normal **CPU** runtime. Do not request a T4. The pharmacy step is Python and the network is small, so a GPU waits on the environment and the free T4 quota runs out before the results can be saved.

A FARUMASI rider collects an order in Kigali and delivers it through traffic and rain, checks the prescription, hands over, and scans the QR code. This notebook runs the whole experiment in one go:

1. **Sweep.** 10 settings for each of DQN, REINFORCE, PPO and A2C, 30,000 frames per setting (40 runs).
2. **Long runs.** The best setting of each algorithm, 120,000 frames, two training seeds (8 runs).
3. **Evaluation.** Every checkpoint is scored on 100 deliveries that no run ever saw, then the figures, tables and `docs/RESULTS.md` are built.

The two CPU cores run two algorithms at a time, so expect roughly **4 to 6 hours** in total.

**How to run it**

1. **Runtime → Run all**, then allow the Google Drive prompt. After every finished run the progress is copied to `MyDrive/farumasi_rl_v4`.
2. Keep the Colab tab open. A closed tab or a sleeping laptop can stop the runtime.
3. If Colab disconnects, open this notebook again and choose **Run all**. Finished runs are restored from Drive and skipped, so nothing is trained twice.
4. When the last cell finishes it downloads `farumasi_rl_results.zip`. Keep that file; it is also copied to Drive.

The training cell prints a line per update with the stream letter (A: DQN and REINFORCE, B: PPO and A2C).

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTHONUNBUFFERED"] = "1"

import torch

print("python", sys.version.split()[0], "torch", torch.__version__, flush=True)
if sys.version_info < (3, 12):
    raise RuntimeError("TorchRL 0.15 needs Python 3.12 or newer. This runtime is " + sys.version.split()[0])

REPO = "https://github.com/Emmanuel-NS/Rl_Mission-based_Summative.git"
base = Path("/content") if Path("/content").exists() else Path.cwd()
dest = base / "Rl_Mission-based_Summative"
if not (dest / "main.py").exists():
    if dest.exists():
        shutil.rmtree(dest)
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(dest)])
os.chdir(dest)
print("working directory", Path.cwd(), flush=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    backup = Path("/content/drive/MyDrive/farumasi_rl_v4")
    backup.mkdir(parents=True, exist_ok=True)
    os.environ["FARUMASI_BACKUP"] = str(backup)
    for name in ("logs", "models"):
        source = backup / name
        if source.exists():
            shutil.copytree(source, dest / name, dirs_exist_ok=True)
            print("restored", name, flush=True)
except Exception as error:
    print("Drive is not mounted, so a disconnect can wipe this session.", error, flush=True)

subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "gymnasium>=1.0", "pygame>=2.6", "cloudpickle", "packaging", "pyvers",
    ]
)
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "torchrl==0.15.0", "tensordict==0.15.6", "hoptorch==0.1.4",
    ]
)

check = """
import torchrl
from torchrl.collectors import Collector
from torchrl.objectives import A2CLoss, ClipPPOLoss, DQNLoss, ReinforceLoss
from training.torchrl_env import make_env
env = make_env(0)
batch = env.reset()
print("ready", tuple(batch.get("observation").shape))
"""
subprocess.check_call([sys.executable, "-u", "-c", check], cwd=dest)


python 3.13.18 torch 2.14.1+cpu
working directory /content/Rl_Mission-based_Summative
Mounted at /content/drive


0

In [2]:
import os
import subprocess
import sys
from pathlib import Path

root = Path("/content/Rl_Mission-based_Summative")
if (root / "main.py").exists():
    os.chdir(root)
subprocess.check_call([sys.executable, "-u", "main.py", "--baselines"], cwd=Path.cwd())
subprocess.check_call([sys.executable, "-u", "main.py", "--screenshot"], cwd=Path.cwd())
print("screenshot", Path("assets/environment.png").resolve(), flush=True)


screenshot /content/Rl_Mission-based_Summative/assets/environment.png


In [3]:
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

root = Path("/content/Rl_Mission-based_Summative")
if (root / "main.py").exists():
    os.chdir(root)

SWEEP_FRAMES = 30_000     # frames per sweep setting (10 settings x 4 algorithms)
FINAL_FRAMES = 120_000    # frames per long run
FINAL_SEEDS = 2           # training seeds per long run
EVAL_EVERY = 5_000
RETRIES = 2               # a failed step is started again; finished runs are skipped

STREAMS = {"A": ["dqn", "reinforce"], "B": ["ppo", "a2c"]}
worker_env = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", PYTHONUNBUFFERED="1")
failures = []


def steps_for(algos):
    return [
        ["main.py", "--sweep", "all", "--algos", *algos, "--device", "cpu", "--frames", str(SWEEP_FRAMES), "--eval-every", str(EVAL_EVERY)],
        ["main.py", "--final", "--algos", *algos, "--device", "cpu", "--frames", str(FINAL_FRAMES), "--seeds", str(FINAL_SEEDS), "--eval-every", str(EVAL_EVERY)],
    ]


def run_stream(name, algos):
    for args in steps_for(algos):
        for attempt in range(RETRIES + 1):
            process = subprocess.Popen(
                [sys.executable, "-u", *args],
                cwd=Path.cwd(), env=worker_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
            )
            for line in process.stdout:
                print(f"[{name}] {line}", end="", flush=True)
            if process.wait() == 0:
                break
            print(f"[{name}] step failed (attempt {attempt + 1}): {' '.join(args)}", flush=True)
            time.sleep(10)
        else:
            failures.append((name, args))
            return
    print(f"[{name}] finished {', '.join(algos)}", flush=True)


started = time.time()
threads = [threading.Thread(target=run_stream, args=(name, algos)) for name, algos in STREAMS.items()]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(f"training took {(time.time() - started) / 3600:.1f} hours", flush=True)
if failures:
    raise RuntimeError(f"these steps did not finish, run this cell again: {failures}")

print("Scoring every checkpoint on 100 unseen deliveries and drawing the figures.", flush=True)
subprocess.check_call([sys.executable, "-u", "main.py", "--evaluate"], cwd=Path.cwd(), env=worker_env)
print("done. Run the next cell to download the results.", flush=True)

Streaming output truncated to the last 5000 lines.
[B] a2c exp 09  frame  65728  loss 0.0048  entropy 0.059  train -0.70
[B] a2c exp 09  frame  65792  loss 0.0033  entropy 0.039  train 0.03
[A] dqn exp 04  frame  21760  eps 0.050  loss 0.0183  train 5.07
[B] a2c exp 09  frame  65856  loss -0.0055  entropy 0.078  train 0.98
[B] a2c exp 09  frame  65920  loss -0.0213  entropy 0.087  train -2.64
[B] a2c exp 09  frame  65984  loss 0.0133  entropy 0.097  train -3.16
[B] a2c exp 09  frame  66048  loss -0.0021  entropy 0.047  train 1.37
[B] a2c exp 09  frame  66112  loss -0.0024  entropy 0.079  train 3.51
[A] dqn exp 04  frame  22016  eps 0.050  loss 0.0232  train 4.66
[B] a2c exp 09  frame  66176  loss -0.0077  entropy 0.104  train 2.44
[B] a2c exp 09  frame  66240  loss -0.0175  entropy 0.070  train 0.86
[B] a2c exp 09  frame  66304  loss -0.0003  entropy 0.016  train -2.78
[B] a2c exp 09  frame  66368  loss 0.0030  entropy 0.042  train -2.63
[B] a2c exp 09  frame  66432  loss -0.0086  entr

In [4]:
import shutil
import zipfile
from pathlib import Path

root = Path("/content/Rl_Mission-based_Summative")
if not (root / "main.py").exists():
    root = Path.cwd()
archive = Path("/content/farumasi_rl_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
    for folder in ("models", "logs", "assets", "docs"):
        base = root / folder
        if not base.exists():
            continue
        for path in base.rglob("*"):
            if path.is_file() and path.name != ".gitkeep":
                handle.write(path, path.relative_to(root).as_posix())
backup = Path("/content/drive/MyDrive/farumasi_rl_v4")
if backup.exists():
    shutil.copy(archive, backup / archive.name)
    print("copied zip to Drive", flush=True)
print("saved", archive, "bytes", archive.stat().st_size, flush=True)
try:
    from google.colab import files
    files.download(str(archive))
except Exception as error:
    print(error)


copied zip to Drive
saved /content/farumasi_rl_results.zip bytes 22946386


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>